# Phase 2 — FinBERT 3-D Sentiment Ablation

Run this notebook top-to-bottom in a fresh Google Colab runtime.

**Important:** this starts from the tracked source dataset:

`dataset/stocknet_final_modeling_set.parquet`

It does **not** require the generated `stocknet_final_modeling_set_phase2.parquet`. The existing Phase 2 notebook loads the tracked parquet, processes `Company_Texts`, and generates its embedding artifacts locally. fileciteturn10file13L1-L20

### Representation change

Old:
`Company_Texts → FinBERT → 768-D mean embedding`

New:
`Company_Texts → FinBERT classifier → 3-D [positive, neutral, negative]`

Everything else stays aligned with the established Phase 2 protocol.

## 1. Clone Repository

In [ ]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/AdityaMelkote3004/capstone.git"
REPO_DIR = "/content/capstone"

if not os.path.exists(REPO_DIR):
    print("Cloning capstone...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("capstone already exists; pulling main...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Working directory:", os.getcwd())

## 2. Install Dependencies

In [ ]:
if os.path.exists("requirements.txt"):
    print("Installing repository requirements...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "transformers", "sentencepiece", "accelerate",
        "pyarrow", "scikit-learn"
    ],
    check=True
)

print("✓ Dependencies ready.")

## 3. Imports, Device and Seed

In [ ]:
import glob
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU unavailable; FinBERT inference will be slow.")

## 4. Load the Tracked Modeling Dataset

Do **not** look for `stocknet_final_modeling_set_phase2.parquet`.

That is a generated local artifact. The source dataset is the tracked 45-column modeling parquet used by the existing Phase 2 notebook. fileciteturn10file13L1-L20

In [ ]:
DATA_PATH = os.path.join(
    REPO_DIR,
    "dataset",
    "stocknet_final_modeling_set.parquet"
)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "Tracked modeling dataset not found: " + DATA_PATH
    )

df = pd.read_parquet(DATA_PATH)

print("Path:", DATA_PATH)
print("Shape:", df.shape)
print("Date:", df["Date"].min(), "→", df["Date"].max())
print("Tickers:", df["Ticker"].nunique())

if len(df) != 26603:
    raise RuntimeError(f"Expected 26,603 rows, got {len(df)}.")

if "Company_Texts" not in df.columns:
    raise KeyError("Company_Texts is missing.")

if "Target" not in df.columns:
    raise KeyError("Target is missing.")

print("✓ Base dataset verified.")

## 5. Existing Global Date Split

In [ ]:
from src.data.stocknet_dataset import split_by_date, compute_norm_stats, normalize

train_df, val_df, test_df = split_by_date(df)

actual = (len(train_df), len(val_df), len(test_df))
expected = (15969, 4359, 6275)

print("Train:", actual[0])
print("Val:  ", actual[1])
print("Test: ", actual[2])

if actual != expected:
    raise RuntimeError(f"Split mismatch: expected {expected}, got {actual}")

print("✓ Original Phase 2 split verified.")

## 6. Structured Features

In [ ]:
PRICE_FEATURES = [
    "Return", "RSI_14", "MACD", "MACD_Signal", "MACD_Hist",
    "Volatility_5", "Volatility_20", "Price_MA5_Ratio",
    "Price_MA10_Ratio", "Price_MA20_Ratio", "Volume_Change",
    "HL_Spread", "MA_5", "MA_10"
]

FUNDAMENTAL_FEATURES = [
    "Revenue", "NetIncome", "TotalAssets", "TotalLiabilities",
    "StockholdersEquity", "EPS", "Cash", "ROA"
]

for col in PRICE_FEATURES + FUNDAMENTAL_FEATURES:
    if col not in df.columns:
        raise KeyError(f"Missing feature: {col}")

print("Price:", len(PRICE_FEATURES))
print("Fundamentals:", len(FUNDAMENTAL_FEATURES))

## 7. Load FinBERT Classification Model

Use the classification head, not the 768-D CLS embedding:

`logits → softmax → [positive, neutral, negative]`

In [ ]:
FINBERT_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(FINBERT_NAME)
finbert = AutoModelForSequenceClassification.from_pretrained(
    FINBERT_NAME
).to(DEVICE)

finbert.eval()

print("Labels:", finbert.config.id2label)
print("Classes:", finbert.config.num_labels)

if finbert.config.num_labels != 3:
    raise RuntimeError("Expected exactly 3 FinBERT classes.")

print("✓ FinBERT ready.")

## 8. Daily 3-D Sentiment

The existing Phase 2 notebook splits `Company_Texts` on `[SEP]`, encodes individual texts and mean-pools them for each stock-day. fileciteturn10file13L1-L20

We keep that aggregation strategy and replace each text embedding with three FinBERT sentiment probabilities.

In [ ]:
def finbert_probabilities(texts, batch_size=32, max_length=256):
    if not texts:
        return np.empty((0, 3), dtype=np.float32)

    outputs = []

    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]

            inputs = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

            logits = finbert(**inputs).logits
            probs = torch.softmax(logits, dim=-1)
            outputs.append(probs.cpu())

    return torch.cat(outputs, dim=0).numpy().astype(np.float32)


def get_daily_sentiment(text):
    if pd.isna(text):
        return np.zeros(3, dtype=np.float32)

    text = str(text).strip()
    if not text:
        return np.zeros(3, dtype=np.float32)

    texts = [x.strip() for x in text.split("[SEP]") if x.strip()]
    if not texts:
        return np.zeros(3, dtype=np.float32)

    return finbert_probabilities(texts).mean(axis=0).astype(np.float32)


example = get_daily_sentiment(df.loc[0, "Company_Texts"])

print("Example shape:", example.shape)
print("Example:", example)
print("Probability sum:", example.sum())

if example.shape != (3,):
    raise RuntimeError("Daily sentiment is not 3-D.")

## 9. Generate 3-D Sentiment for Every Stock-Day

This is the expensive cell. GPU is used automatically if available.

In [ ]:
sentiment = np.zeros((len(df), 3), dtype=np.float32)

texts = df["Company_Texts"].fillna("").astype(str).tolist()

ROW_CHUNK = 128

for start in range(0, len(texts), ROW_CHUNK):
    end = min(start + ROW_CHUNK, len(texts))

    for i in range(start, end):
        sentiment[i] = get_daily_sentiment(texts[i])

    if start == 0 or start % (ROW_CHUNK * 10) == 0:
        print(f"Processed {end:,}/{len(texts):,}")

if not np.isfinite(sentiment).all():
    raise RuntimeError("NaN/Inf found in sentiment.")

print("Shape:", sentiment.shape)
print("NaN:", np.isnan(sentiment).sum())
print("Inf:", np.isinf(sentiment).sum())
print(sentiment[:5])

## 10. Save the Local 3-D Representation

In [ ]:
SENTIMENT_FEATURES = [
    "FinBERT_Positive",
    "FinBERT_Neutral",
    "FinBERT_Negative"
]

SENTIMENT_PATH = os.path.join(
    REPO_DIR,
    "finbert_sentiment_3d.npy"
)

np.save(SENTIMENT_PATH, sentiment)

print("Saved:", SENTIMENT_PATH)

## 11. Attach Sentiment to the Original Splits

In [ ]:
df_sent = df.copy()

for i, col in enumerate(SENTIMENT_FEATURES):
    df_sent[col] = sentiment[:, i]

train_sent = df_sent.loc[train_df.index].copy()
val_sent = df_sent.loc[val_df.index].copy()
test_sent = df_sent.loc[test_df.index].copy()

if (len(train_sent), len(val_sent), len(test_sent)) != (15969, 4359, 6275):
    raise RuntimeError("Sentiment attachment changed split sizes.")

print("✓ Alignment preserved.")

## 12. Train-Only Normalization

In [ ]:
STRUCTURED_FEATURES = PRICE_FEATURES + FUNDAMENTAL_FEATURES

means, stds = compute_norm_stats(
    train_sent,
    STRUCTURED_FEATURES
)

train_sent = normalize(train_sent, STRUCTURED_FEATURES, means, stds)
val_sent = normalize(val_sent, STRUCTURED_FEATURES, means, stds)
test_sent = normalize(test_sent, STRUCTURED_FEATURES, means, stds)

print("✓ Structured features normalized using training statistics only.")

## 13. Original W=5 Windows

Expected:

- Train: 15,534
- Validation: 3,924
- Test: 5,840

The project documents W=5 and these exact counts. fileciteturn10file15L1-L20

In [ ]:
class StockNetWindowDataset(Dataset):
    def __init__(self, frame, feature_columns, window_size=5):
        frame = frame.sort_values(
            ["Ticker", "Date"]
        ).reset_index(drop=True)

        self.samples = []

        for _, group in frame.groupby("Ticker", sort=False):
            group = group.sort_values("Date").reset_index(drop=True)

            for i in range(window_size, len(group)):
                x = group.iloc[i-window_size:i][feature_columns].to_numpy(
                    dtype=np.float32
                )
                y = int(group.iloc[i]["Target"])

                self.samples.append(
                    (
                        torch.tensor(x, dtype=torch.float32),
                        torch.tensor(y, dtype=torch.long)
                    )
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


WINDOW_SIZE = 5
ALL_FEATURES = STRUCTURED_FEATURES + SENTIMENT_FEATURES

train_full = StockNetWindowDataset(train_sent, ALL_FEATURES, WINDOW_SIZE)
val_full = StockNetWindowDataset(val_sent, ALL_FEATURES, WINDOW_SIZE)
test_full = StockNetWindowDataset(test_sent, ALL_FEATURES, WINDOW_SIZE)

counts = (len(train_full), len(val_full), len(test_full))

print("Train windows:", counts[0])
print("Val windows:", counts[1])
print("Test windows:", counts[2])

if counts != (15534, 3924, 5840):
    raise RuntimeError(
        f"Window mismatch: expected (15534, 3924, 5840), got {counts}"
    )

print("✓ Original W=5 counts verified.")

## 14. Four Ablations

In [ ]:
EXPERIMENTS = {
    "A_Price": PRICE_FEATURES,
    "B_Price_Fundamentals": PRICE_FEATURES + FUNDAMENTAL_FEATURES,
    "C_Price_FinBERT_Sentiment": PRICE_FEATURES + SENTIMENT_FEATURES,
    "D_Price_Fundamentals_FinBERT_Sentiment":
        PRICE_FEATURES + FUNDAMENTAL_FEATURES + SENTIMENT_FEATURES
}

for name, columns in EXPERIMENTS.items():
    print(name, "→", len(columns), "features")

In [ ]:
class SelectedDataset(Dataset):
    def __init__(self, full_dataset, all_columns, selected_columns):
        self.full = full_dataset
        self.indices = [all_columns.index(c) for c in selected_columns]

    def __len__(self):
        return len(self.full)

    def __getitem__(self, index):
        x, y = self.full[index]
        return x[:, self.indices], y


experiment_datasets = {}

for name, columns in EXPERIMENTS.items():
    experiment_datasets[name] = {
        "train": SelectedDataset(train_full, ALL_FEATURES, columns),
        "val": SelectedDataset(val_full, ALL_FEATURES, columns),
        "test": SelectedDataset(test_full, ALL_FEATURES, columns)
    }

print("✓ Four experiment datasets ready.")

## 15. LSTM Baseline

Established settings: Adam, LR 0.001, weight decay 1e-4, batch size 64, max 50 epochs, patience 10, validation MCC early stopping, gradient clipping 1.0, seed 42. fileciteturn10file15L1-L20

In [ ]:
class StockNetLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.head(h_n[-1])


def evaluate(model, loader):
    model.eval()

    y_true = []
    y_pred = []
    y_prob = []

    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(DEVICE))

            y_true.extend(y.numpy())
            y_pred.extend(logits.argmax(1).cpu().numpy())
            y_prob.extend(
                torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            )

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_prob)
    }

In [ ]:
def train_experiment(bundle, input_dim, max_epochs=50, patience=10):
    train_loader = DataLoader(
        bundle["train"], batch_size=64, shuffle=True
    )
    val_loader = DataLoader(
        bundle["val"], batch_size=64, shuffle=False
    )
    test_loader = DataLoader(
        bundle["test"], batch_size=64, shuffle=False
    )

    model = StockNetLSTM(input_dim).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001,
        weight_decay=1e-4
    )

    criterion = nn.CrossEntropyLoss()

    best_mcc = -float("inf")
    best_state = None
    wait = 0

    for epoch in range(1, max_epochs + 1):
        model.train()

        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()

            logits = model(x)
            loss = criterion(logits, y)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            optimizer.step()

        val_metrics = evaluate(model, val_loader)

        if val_metrics["mcc"] > best_mcc:
            best_mcc = val_metrics["mcc"]
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            wait = 0
        else:
            wait += 1

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"Ep {epoch:3d} | "
                f"Val Acc={val_metrics['accuracy']:.3f} "
                f"F1={val_metrics['f1']:.3f} "
                f"MCC={val_metrics['mcc']:.4f} "
                f"AUC={val_metrics['auc']:.4f}"
            )

        if wait >= patience:
            print("Early stop at epoch", epoch)
            break

    if best_state is None:
        raise RuntimeError("No best checkpoint was created.")

    model.load_state_dict(best_state)

    return (
        model,
        evaluate(model, val_loader),
        evaluate(model, test_loader)
    )

## 16. Run All Four Experiments

In [ ]:
results = {}
models = {}

for name, columns in EXPERIMENTS.items():
    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    set_seed()

    model, val_metrics, test_metrics = train_experiment(
        experiment_datasets[name],
        len(columns)
    )

    models[name] = model

    results[name] = {
        "validation": val_metrics,
        "test": test_metrics
    }

    print("Validation:", val_metrics)
    print("Test:", test_metrics)

## 17. Results

In [ ]:
rows = []

for name, result in results.items():
    rows.append({
        "Experiment": name,
        "Val_Accuracy": result["validation"]["accuracy"],
        "Val_F1": result["validation"]["f1"],
        "Val_MCC": result["validation"]["mcc"],
        "Val_AUC": result["validation"]["auc"],
        "Test_Accuracy": result["test"]["accuracy"],
        "Test_F1": result["test"]["f1"],
        "Test_MCC": result["test"]["mcc"],
        "Test_AUC": result["test"]["auc"]
    })

results_df = pd.DataFrame(rows).sort_values(
    ["Val_MCC", "Val_AUC"],
    ascending=False
)

display(results_df)

## 18. Select on Validation, Then Report Test

In [ ]:
ranking = sorted(
    results.items(),
    key=lambda item: (
        item[1]["validation"]["mcc"],
        item[1]["validation"]["auc"]
    ),
    reverse=True
)

for rank, (name, result) in enumerate(ranking, 1):
    v = result["validation"]
    print(
        f"{rank}. {name} | "
        f"MCC={v['mcc']:.4f} | "
        f"AUC={v['auc']:.4f}"
    )

BEST_EXPERIMENT = ranking[0][0]

print("\nSelected:", BEST_EXPERIMENT)
print("\nFINAL TEST RESULT")

for metric, value in results[BEST_EXPERIMENT]["test"].items():
    print(f"{metric.upper():>10}: {value:.4f}")

## 19. Previous 768-D FinBERT Benchmark

The existing Phase 2 notebook reports:

| Feature Set | Accuracy | F1 | MCC | AUC |
|---|---:|---:|---:|---:|
| Price + Company FinBERT | 0.5248 | 0.4721 | 0.0459 | 0.5304 |
| Price + Fundamentals + Company FinBERT | 0.4870 | 0.6550 | 0.0000 | 0.5270 |

These are the benchmarks this ablation is intended to compare against. fileciteturn10file10L1-L20

## 20. Save Results + Generate Markdown

In [ ]:
RESULT_DIR = os.path.join(
    REPO_DIR,
    "results",
    "phase2_finbert_sentiment"
)

os.makedirs(RESULT_DIR, exist_ok=True)

csv_path = os.path.join(RESULT_DIR, "results.csv")
json_path = os.path.join(RESULT_DIR, "results.json")
md_path = os.path.join(RESULT_DIR, "phase2_finbert_sentiment.md")

results_df.to_csv(csv_path, index=False)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

report = [
    "# Phase 2 — FinBERT 3-D Sentiment Ablation",
    "",
    "## Objective",
    "",
    "Replace the 768-D Company FinBERT representation with three FinBERT sentiment probabilities while retaining the established Phase 2 data and evaluation protocol.",
    "",
    "## Data Protocol",
    "",
    "| Quantity | Value |",
    "|---|---:|",
    "| Train rows | 15,969 |",
    "| Validation rows | 4,359 |",
    "| Test rows | 6,275 |",
    "| Train windows | 15,534 |",
    "| Validation windows | 3,924 |",
    "| Test windows | 5,840 |",
    "| Window size | 5 |",
    "",
    "## Representation",
    "",
    "Company_Texts are split on [SEP]. Each text is passed through ProsusAI/FinBERT's classification head. Softmax produces Positive, Neutral and Negative probabilities, which are mean-pooled into one 3-D stock-day representation.",
    "",
    "## Experiments",
    "",
    "| Experiment | Input |",
    "|---|---|",
    "| A | Price |",
    "| B | Price + Fundamentals |",
    "| C | Price + FinBERT Sentiment |",
    "| D | Price + Fundamentals + FinBERT Sentiment |",
    "",
    "## Results",
    "",
    results_df.to_markdown(index=False),
    "",
    "## Selected Configuration",
    "",
    f"**{BEST_EXPERIMENT}** selected using validation MCC, with validation AUC as secondary criterion.",
    "",
    "## Final Test Result",
    "",
    "| Metric | Value |"
]

for metric, value in results[BEST_EXPERIMENT]["test"].items():
    report.append(f"| {metric.upper()} | {value:.4f} |")

report += [
    "",
    "## Previous 768-D FinBERT References",
    "",
    "| Feature Set | Accuracy | F1 | MCC | AUC |",
    "|---|---:|---:|---:|---:|",
    "| Price + Company FinBERT | 0.5248 | 0.4721 | 0.0459 | 0.5304 |",
    "| Price + Fundamentals + Company FinBERT | 0.4870 | 0.6550 | 0.0000 | 0.5270 |",
    "",
    "## Interpretation",
    "",
    "This is a controlled representation ablation. The source dataset, chronological split, train-only normalization, W=5 windows and LSTM training/evaluation protocol are retained."
]

Path(md_path).write_text("\n".join(report), encoding="utf-8")

print("Saved:")
print(csv_path)
print(json_path)
print(md_path)

## 21. Final Verification

In [ ]:
print("=" * 80)
print("FINAL VERIFICATION")
print("=" * 80)

for label, path in [
    ("Source dataset", DATA_PATH),
    ("3-D sentiment", SENTIMENT_PATH),
    ("CSV results", csv_path),
    ("JSON results", json_path),
    ("Markdown results", md_path)
]:
    print(
        "✓" if os.path.exists(path) else "✗",
        label,
        "→",
        path
    )

print("\nExpected rows: 15969 / 4359 / 6275")
print("Expected windows: 15534 / 3924 / 5840")

print("\nGit status:")
subprocess.run(["git", "-C", REPO_DIR, "status", "--short"])